Частина 2. Аналіз датасету Individual Household Electric Power Consumption
Завдання 1: 
Звантажити та відкрити датасет.Здійснити data cleaning (обробка пропущених значень)


In [2]:
import pandas as pd
import numpy as np

df_power = pd.read_csv('household_power_consumption.txt', sep=';', na_values=['?'], low_memory=False)
df_power = df_power.dropna()
df_power['Datetime'] = pd.to_datetime(df_power['Date'] + ' ' + df_power['Time'], format='%d/%m/%Y %H:%M:%S')
df_power = df_power.drop(['Date', 'Time'], axis=1)
cols = ['Datetime'] + [col for col in df_power.columns if col != 'Datetime']
df_power = df_power[cols]
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
for col in numeric_cols:
    df_power[col] = pd.to_numeric(df_power[col])

print(f"Датасет успішно завантажено та очищено! Залишилось записів: {len(df_power)}")
display(df_power.head())

Датасет успішно завантажено та очищено! Залишилось записів: 2049280


,Datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


Завдання 2: Окремими функціями сформувати вибірки. 
Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [5]:
def filter_high_active_power(df, threshold=5.0):
    """Повертає всі записи, де Global_active_power > threshold."""
    return df[df['Global_active_power'] > threshold]

high_power_df = filter_high_active_power(df_power)
print(f"=== Записи, де активна потужність > 5 кВт ===")
print(f"Знайдено рядків: {len(high_power_df)}")
display(high_power_df.head())

print("\n Профілювання часу виконання ")
%timeit filter_high_active_power(df_power)

=== Записи, де активна потужність > 5 кВт ===
Знайдено рядків: 17547


,Datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,2006-12-16 17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,2006-12-16 17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0



 Профілювання часу виконання 
3.11 ms ± 124 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Завдання 3: Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.

In [4]:
def filter_current_and_appliances(df):
    """
    Відбирає записи, де сила струму 19-20 А, 
    і споживання пральної машини/холодильника (Sub_metering_2) 
    більше за споживання бойлера/кондиціонера (Sub_metering_3).
    """
    current_filtered = df[(df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)]

    result = current_filtered[current_filtered['Sub_metering_2'] > current_filtered['Sub_metering_3']]
    
    return result

appliances_df = filter_current_and_appliances(df_power)

print(f" Записи: струм 19-20 А та пралка/холодильник > бойлер/кондиціонер ")
print(f"Знайдено рядків: {len(appliances_df)}")
display(appliances_df.head())

print("\n Профілювання часу виконання ")
%timeit filter_current_and_appliances(df_power)

 Записи: струм 19-20 А та пралка/холодильник > бойлер/кондиціонер 
Знайдено рядків: 2509


,Datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,2006-12-16 18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,2006-12-17 01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,2006-12-17 01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,2006-12-17 01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,2006-12-17 01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0



 Профілювання часу виконання 
5.25 ms ± 87 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Завдання 4: Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [7]:
def sample_and_average_consumption(df, n_samples=500000):
    """
    Обирає випадкові записи без повторень та обчислює 
    середнє для Sub_metering_1, Sub_metering_2 та Sub_metering_3.
    """
    sample_df = df.sample(n=n_samples, replace=False, random_state=42)

    means = {
        'Група 1 (Кухня)': sample_df['Sub_metering_1'].mean(),
        'Група 2 (Пралка/Холодильник)': sample_df['Sub_metering_2'].mean(),
        'Група 3 (Бойлер/Кондиціонер)': sample_df['Sub_metering_3'].mean()
    }
    
    return means

averages = sample_and_average_consumption(df_power)

print(" Середні величини споживання для 500,000 випадкових записів ")
for group, mean_val in averages.items():
    print(f"{group}: {mean_val:.4f} Вт-год")

print("\n Профілювання часу виконання ")
%timeit sample_and_average_consumption(df_power)

 Середні величини споживання для 500,000 випадкових записів 
Група 1 (Кухня): 1.1193 Вт-год
Група 2 (Пралка/Холодильник): 1.3089 Вт-год
Група 3 (Бойлер/Кондиціонер): 6.4530 Вт-год

 Профілювання часу виконання 
117 ms ± 1.85 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Завдання 5: Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому. Серед відібраних визначити ті, у яких група 2 є найбільшою. Обрати кожен 3-й результат із першої половини та кожен 4-й результат із другої половини.

In [9]:
def filter_complex_evening_consumption(df):
    """
    Виконує комплексну вибірку: час > 18:00, потужність > 6 кВт, 
    Група 2 найбільша, і специфічне зрізання по половинах.
    """

    evening_df = df[df['Datetime'].dt.hour >= 18]

    high_power = evening_df[evening_df['Global_active_power'] > 6]

    group2_max = high_power[
        (high_power['Sub_metering_2'] > high_power['Sub_metering_1']) & 
        (high_power['Sub_metering_2'] > high_power['Sub_metering_3'])
    ]

    half_idx = len(group2_max) // 2
    first_half = group2_max.iloc[:half_idx]
    second_half = group2_max.iloc[half_idx:]

    res_first = first_half.iloc[::3]
    res_second = second_half.iloc[::4]

    return pd.concat([res_first, res_second])

complex_result_df = filter_complex_evening_consumption(df_power)

print(" Складна вечірня вибірка ")
print(f"Знайдено фінальних рядків: {len(complex_result_df)}")
display(complex_result_df.head(10))

print("\n Профілювання часу виконання ")
%timeit filter_complex_evening_consumption(df_power)

 Складна вечірня вибірка 
Знайдено фінальних рядків: 310


,Datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,2006-12-16 18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,2006-12-16 18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,2006-12-28 20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,2006-12-28 21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,2006-12-28 21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0
17504,2006-12-28 21:08:00,7.352,0.000,235.45,31.2,1.0,73.0,17.0
17507,2006-12-28 21:11:00,9.048,0.000,231.48,39.0,34.0,71.0,16.0
17510,2006-12-28 21:14:00,9.118,0.108,231.18,39.4,36.0,72.0,16.0
17513,2006-12-28 21:17:00,7.040,0.130,233.27,30.2,37.0,38.0,17.0
18952,2006-12-29 21:16:00,6.146,0.116,230.53,26.6,0.0,70.0,0.0



 Профілювання часу виконання 
66.7 ms ± 2.88 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Завдання 6: Пронормувати та стандартизувати датасет. Підрахувати коефіцієнти Пірсона та Спірмена для двох атрибутів. Провести One Hot Encoding категоріального атрибута.

In [12]:
%pip install scipy

   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ------ --------------------------------- 5.8/36.5 MB 35.1 MB/s eta 0:00:01
   ---------------- ----------------------- 15.5/36.5 MB 40.5 MB/s eta 0:00:01
   -------------------------- ------------- 24.1/36.5 MB 41.3 MB/s eta 0:00:01
   -------------------------------------- - 35.4/36.5 MB 43.2 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 41.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
df_sample = df_power.sample(10000, random_state=42).copy()

numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
df_numeric = df_sample[numeric_cols]

print(" 1. Нормалізація (Min-Max) ")

df_normalized = (df_numeric - df_numeric.min()) / (df_numeric.max() - df_numeric.min())
display(df_normalized.head(3))

print("\n 2. Стандартизація (Z-score) ")

df_standardized = (df_numeric - df_numeric.mean()) / df_numeric.std()
display(df_standardized.head(3))

print("\n 3. Коефіцієнти Пірсона та Спірмена ")
col1, col2 = 'Global_active_power', 'Global_intensity'
pearson_corr = df_sample[col1].corr(df_sample[col2], method='pearson')
spearman_corr = df_sample[col1].corr(df_sample[col2], method='spearman')

print(f"Кореляція між '{col1}' та '{col2}':")
print(f"Пірсон (лінійна залежність): {pearson_corr:.4f}")
print(f"Спірмен (монотонна залежність): {spearman_corr:.4f}")

print("\n 4. One Hot Encoding ")
def get_time_of_day(hour):
    if 6 <= hour < 12: return 'Morning'
    elif 12 <= hour < 18: return 'Afternoon'
    elif 18 <= hour < 24: return 'Evening'
    else: return 'Night'
    
df_sample['Time_Category'] = df_sample['Datetime'].dt.hour.apply(get_time_of_day)

df_ohe = pd.get_dummies(df_sample, columns=['Time_Category'], prefix='Is')

ohe_cols = [col for col in df_ohe.columns if 'Is_' in col]
display(df_ohe[['Datetime', 'Global_active_power'] + ohe_cols].head())

 1. Нормалізація (Min-Max) 


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1030580,0.168282,0.082960,0.523291,0.173184,0.0,0.000000,0.600000
1815,0.034980,0.295964,0.723516,0.044693,0.0,0.025974,0.000000
1295977,0.064051,0.336323,0.511270,0.078212,0.0,0.012987,0.033333



 2. Стандартизація (Z-score) 


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1030580,0.364855,-0.456081,-0.211628,0.375171,-0.187757,-0.229421,1.341881
1815,-0.683857,1.223639,1.434525,-0.642488,-0.187757,0.093207,-0.776302
1295977,-0.455148,1.541902,-0.310459,-0.377012,-0.187757,-0.068107,-0.658626



 3. Коефіцієнти Пірсона та Спірмена 
Кореляція між 'Global_active_power' та 'Global_intensity':
Пірсон (лінійна залежність): 0.9989
Спірмен (монотонна залежність): 0.9955

 4. One Hot Encoding 


,Datetime,Global_active_power,Is_Afternoon,Is_Evening,Is_Morning,Is_Night
1030580,2008-12-01 09:44:00,1.502,False,False,True,False
1815,2006-12-17 23:39:00,0.374,False,True,False,False
1295977,2009-06-03 17:01:00,0.620,True,False,False,False
206669,2007-05-09 05:53:00,0.280,False,False,False,True
1048893,2008-12-14 02:57:00,1.372,False,False,False,True
